# A promise to the programmer: Python `assert`
**Python Foundation · Lawson Dong · Lesson 08**

An executable Google Colab lesson about **assertions, failures, testing, and debugging**.

**How to use:** Open in Colab and select **Runtime → Run all**, or run cells individually. Every intentional `AssertionError` in this lesson is caught, so *Run all* can finish. Edit values and run again to test your predictions.

> **Mental model:** `assert condition, message` means "this condition should already be true; if not, something is wrong." It is for debugging assumptions, **not** for security or essential input validation.

## 1. When the condition is true

The expression is evaluated and execution continues. A passing assertion prints **nothing by itself**.

In [ ]:
x = 10
assert x > 0
print("The assertion passed; execution continued.")

# Change the condition to x < 0 and rerun to see AssertionError.

## 2. When the condition is false

A false assertion raises `AssertionError`. Here we **catch** the exception so the notebook can run from top to bottom.

In [ ]:
x = 10
try:
    assert x < 0
except AssertionError as error:
    print(f"{type(error).__name__} raised")
    print("The condition was False.")

# A message after a comma is optional:
try:
    assert x < 0, "x must be negative"
except AssertionError as error:
    print(f"{type(error).__name__}: {error}")

### What actually happens?

| Statement | Outcome |
|---|---|
| `assert True` | No output; keep running |
| `assert False` | Raise `AssertionError` |
| `assert False, "reason"` | Raise `AssertionError("reason")` |

Conceptually, **when assertions are enabled**, `assert condition, message` acts similarly to:

```python
if __debug__:
    if not condition:
        raise AssertionError(message)
```

The message expression is only evaluated if the assertion fails. The real `assert` statement may be removed by optimization; the explanatory `if` code above is *not* a replacement.

## 3. Compare `assert` with `if ...: raise`

Use assertions to document internal assumptions during development. Use an explicit exception for a condition that must **always** be checked, such as user input.

In [ ]:
def average(numbers):
    if not numbers:                    # Input validation must always run
        raise ValueError("numbers cannot be empty")
    result = sum(numbers) / len(numbers)
    assert isinstance(result, (int, float))  # Internal sanity check
    return result

print("average([2, 4, 6]) =", average([2, 4, 6]))

try:
    average([])
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

## 4. Assertions as algorithm tests: binary search

This example uses **input → expected output** checks. Start with easy cases, then boundary cases: first element, last element, missing target, singleton list, and empty list. A correct function passes silently; the last `print` is our own message.

In [ ]:
def binary_search(nums, target):
    left, right = 0, len(nums) - 1
    while left <= right:
        mid = (left + right) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

assert binary_search([1, 3, 5, 7], 5) == 2   # Found in middle
assert binary_search([1, 3, 5, 7], 1) == 0   # First
assert binary_search([1, 3, 5, 7], 7) == 3   # Last
assert binary_search([1, 3, 5, 7], 8) == -1  # Missing
assert binary_search([9], 9) == 0           # Singleton
assert binary_search([], 3) == -1           # Empty list
print("All six test cases passed.")

## 5. Inspect a failing test without stopping "Run all"

Suppose a buggy search always returns `-1`. A good test points directly at the incorrect assumption.

In [ ]:
def buggy_search(nums, target):
    return -1

try:
    actual = buggy_search([1, 3, 5, 7], 5)
    expected = 2
    assert actual == expected, f"expected {expected}, got {actual}"
except AssertionError as error:
    print(f"TEST FAILED — {error}")

# Try changing buggy_search to call binary_search and rerun.

## 6. Interactive assertion explorer

Move the slider or select "Allow zero", then click **Test assertion**. This runs a real Python `assert`, catches the expected failure, and shows the result. No manual Python editing is required, but the code remains editable.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # Also runs in Jupyter outside Colab.

number = widgets.IntSlider(value=2, min=-5, max=5, step=1, description="x:")
allow_zero = widgets.Checkbox(value=False, description="Allow zero")
run = widgets.Button(description="Test assertion", button_style="info")
result = widgets.Output()

def test_assertion(_=None):
    with result:
        result.clear_output()
        x = number.value
        try:
            if allow_zero.value:
                assert x >= 0, "x must be non-negative"
            else:
                assert x > 0, "x must be strictly positive"
            print(f"PASS: x = {x} satisfies the condition.")
        except AssertionError as error:
            print(f"FAIL: AssertionError: {error} (x = {x})")

run.on_click(test_assertion)
display(widgets.VBox([widgets.HBox([number, allow_zero]), run, result]))
test_assertion()

## 7. Important: `python -O` skips assertions

Python's optimization flag removes ordinary `assert` statements (and makes `__debug__` false). **Never** put required validation, safety checks, or side effects inside assertions.

Below we launch tiny separate Python processes. The first uses normal mode; the second uses `-O`.

In [ ]:
import subprocess
import sys

program = 'assert False, "check was executed"; print("Reached the end")'

for optimized in (False, True):
    command = [sys.executable] + (["-O"] if optimized else []) + ["-c", program]
    completed = subprocess.run(command, capture_output=True, text=True)
    print("Mode:", "optimized (-O)" if optimized else "normal")
    print("Exit code:", completed.returncode)
    print("Contains AssertionError:", "AssertionError" in completed.stderr)
    print("stdout:", completed.stdout.strip() or "(none)")
    print()

## 8. Predict first, then test

1. What will `assert 3 == 3` display?
2. What error does `assert 3 == 4, "not equal"` raise?
3. Will `assert binary_search([], 7) == -1` pass?
4. Should a login check use `assert password_is_valid`? **No:** it can be removed with `-O`. Use ordinary conditional validation.

Make your own tests below. Keep unexpected failures visible during actual development; the `try/except` wrapper here is only for this teaching exercise.

In [ ]:
tests = [
    ("numbers match", lambda: 3 == 3),
    ("empty search", lambda: binary_search([], 7) == -1),
    ("wrong prediction", lambda: 3 == 4),
]

for name, condition in tests:
    try:
        assert condition(), f"{name}: predicted True, got False"
        print("PASS:", name)
    except AssertionError as error:
        print("FAIL:", error)

## Key takeaways

- `assert condition` checks a programmer's assumption **while assertions are enabled**.
- A failure raises `AssertionError`; `assert condition, message` gives context.
- Good assertion tests include **boundary cases**, not just normal inputs.
- Use `if ...: raise ValueError(...)` (or another appropriate exception) for mandatory checks.
- `assert` is **not guaranteed to execute** under `python -O`.

**References:** [Python language reference: the assert statement](https://docs.python.org/3/reference/simple_stmts.html#the-assert-statement) · [Python command line: -O](https://docs.python.org/3/using/cmdline.html#cmdoption-O) · [Python: built-in exceptions](https://docs.python.org/3/library/exceptions.html#AssertionError)